In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import *

# initial load
c_schema = StructType([
    StructField('customer_id',IntegerType(),False),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType()),
])

a_schema = StructType([
    StructField('account_id',IntegerType()),
    StructField('customer_id',IntegerType(),False),
    StructField('account_type',StringType()),
    StructField('branch_city',StringType()),
    StructField('status',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType())
])

t_schema = StructType(
    [
        StructField('transaction_id',IntegerType()),
        StructField('account_id',IntegerType()),
        StructField('transaction_type',StringType()),
        StructField('amount',IntegerType()),
        StructField('transaction_date',StringType())
    ]
)

bcustomers = spark.read.format('csv').option('header','true').schema(c_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoGwithSCD2andMergeSQL/customerDim.csv')

baccounts = spark.read.format('csv').option('header','true').schema(a_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoGwithSCD2andMergeSQL/accountDim.csv')

btransactions = spark.read.format('csv').option('header','true').schema(t_schema).load('/Workspace/Users/manikantak177@gmail.com/BtoGwithSCD2andMergeSQL/transactionFact.csv')

# drop records if null
bcustomers = bcustomers.dropna(subset=('customer_id'))
baccounts = baccounts.dropna(subset=('customer_id','account_id'))
btransactions = btransactions.dropna(subset=('account_id'))

# removeduplicates l1
bcustomers = bcustomers.dropDuplicates()
baccounts = baccounts.dropDuplicates()
btransactions = btransactions.dropDuplicates()

# null handling
bcustomers = bcustomers.fillna({
     'customer_name':'Unknown',
     'city':'Unknown',
     'customer_segment':'Unknown',
     'email':'Unknown',
     'phone':'Unknown',
     'is_current':'Unknown'
})

baccounts = baccounts.fillna({
    'account_type':'Unknown',
    'branch_city':'Unknown',
    'status':'Unknown',
    'is_current':'Unknown'
}
)

btransactions = btransactions.fillna({
    'transaction_type':'Unknown'
})

# standardization
phone_regex = r'[^0-9]'
bcustomers = bcustomers.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.trim(f.col('email'))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,'')).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

baccounts = baccounts.withColumn('account_type',f.initcap(f.trim(f.col('account_type')))).withColumn('branch_city',f.initcap(f.trim('branch_city'))).withColumn('status',f.initcap(f.trim('status'))).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

btransactions = btransactions.withColumn('transaction_type',f.initcap(f.trim(f.col('transaction_type')))).orderBy(f.col('account_id'))

# data validation 
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
phone_validation = (f.length(f.col('phone'))==10)
email_validation = (f.col('email').rlike(email_regex))
bcustomers = bcustomers.withColumn('dataValidation',f.when((phone_validation) & (email_validation),'Pass').otherwise('Fail'))

account_validation = (f.col('status') == "Active")
baccounts = baccounts.withColumn('dataValidation',f.when((account_validation) & ((f.col('account_type') == 'Savings') | (f.col('account_type') == 'Current')),'Pass').otherwise('Fail'))

amount_validation = (f.col('amount') >= 0)
btransactions = btransactions.withColumn('dataValidation',f.when((amount_validation),'Pass').otherwise('Fail'))

# error records
silver_customers = bcustomers.filter(f.col('dataValidation') == 'Pass')
silver_accounts = baccounts.filter(f.col('dataValidation')=='Pass')
silver_transactions = btransactions.filter(f.col('dataValidation') == 'Pass')

error_customers = bcustomers.filter(f.col('dataValidation') == 'Fail')
error_accounts = baccounts.filter(f.col('dataValidation')=='Fail')
error_transactions = btransactions.filter(f.col('dataValidation') == 'Fail')

# audit columns 
silver_customers = silver_customers.withColumn('updated_customer_dt',f.lit('9999-12-31')).withColumn('created_customer_dt',f.current_date().cast("string")).drop('dataValidation')
silver_accounts = silver_accounts.withColumn('updated_account_dt',f.lit('9999-12-31')).withColumn('created_account_dt',f.current_date().cast("string")).drop('dataValidation','start_date','end_date','is_current')
silver_transactions = silver_transactions.withColumn('updated_transaction_dt',f.lit('9999-12-31')).withColumn('created_transaction_dt',f.current_date().cast("string")).drop('dataValidation')

# look up table
# gold = silver_accounts.alias('a').join(silver_customers.alias('a'),on='customer_id',how='left').join(silver_transactions.alias('t'),on='account_id',how='left').select('*').orderBy(f.col('customer_id'))

(silver_customers.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madreporting.customers'))
(silver_accounts.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madreporting.accounts'))
(silver_transactions.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madreporting.transactions'))

# (gold.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madreporting.gold'))
# gold.show()


